# convert solar data from model into format for Homer

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

INPUT_FILE = "../data/tmy_2024_NYC.csv"      # update path if needed
OUTPUT_DIR = "homer"
MATCH_LEAP_YEAR = False               # True: 8784 hours (2024 calendar, matches your HOMER project); False: 8760

def ts():
    """Timestamp string for filenames, e.g. 20260923_143005"""
    return datetime.now().strftime("%Y%m%d_%H%M%S")

In [7]:
weather = pd.read_csv(INPUT_FILE, index_col=0)
weather.index = pd.to_datetime(weather.index)

assert len(weather) == 8760, f"Expected 8760 hourly rows, got {len(weather)}"
assert weather["ghi"].isna().sum() == 0, "Missing GHI values found"

first = weather.iloc[0]
assert (first["Month"], first["Day"], first["Hour"]) == (1, 1, 0), "File must start Jan 1, hour 0"

print(f"First row: {weather.index[0]}  (UTC offset {weather.index[0].strftime('%z')})")
print(f"Last row:  {weather.index[-1]}")
print("Month -> source year:", weather.groupby("Month")["Year"].first().to_dict())

First row: 2013-01-01 00:30:00-05:00  (UTC offset -0500)
Last row:  2017-12-31 23:30:00-05:00
Month -> source year: {1: 2013, 2: 2005, 3: 2011, 4: 2022, 5: 2006, 6: 2004, 7: 2005, 8: 2006, 9: 2012, 10: 2022, 11: 2017, 12: 2017}


In [8]:
ghi_kw_m2 = (weather["ghi"].clip(lower=0) / 1000.0).to_numpy()   # W/m² -> kW/m²

if MATCH_LEAP_YEAR:
    feb28 = ghi_kw_m2[58*24 : 59*24]                    # day 59 of the year = Feb 28
    ghi_kw_m2 = np.insert(ghi_kw_m2, 59*24, feb28)      # insert a copy as Feb 29

n_days = len(ghi_kw_m2) // 24
print(f"Rows: {len(ghi_kw_m2)} ({n_days} days, hourly)")

Rows: 8760 (365 days, hourly)


In [9]:
profile = pd.Series(ghi_kw_m2).groupby(np.arange(len(ghi_kw_m2)) % 24).mean()

# one daylight run (consecutive nonzero hours) per day confirms hourly data with no shift
nz = np.r_[0, (ghi_kw_m2 > 0).astype(int), 0]
edges = np.flatnonzero(np.diff(nz))
run_lengths = edges[1::2] - edges[0::2]

print(f"Peak hour of average day: {profile.idxmax()}  (expect 11-12 for NYC standard time)")
print(f"Daylight runs: {len(run_lengths)} (expect {n_days}), length {run_lengths.min()}-{run_lengths.max()} h")
print(f"Annual average: {ghi_kw_m2.sum() / n_days:.3f} kWh/m²/day")

Peak hour of average day: 11  (expect 11-12 for NYC standard time)
Daylight runs: 365 (expect 365), length 9-14 h
Annual average: 4.063 kWh/m²/day


In [10]:
out_path = f"outputs/homer_solar_ghi_{len(ghi_kw_m2)}h_{ts()}.txt"

pd.Series(ghi_kw_m2).to_csv(out_path, index=False, header=False, float_format="%.6f")
print(f"Saved: {out_path}")

Saved: outputs/homer_solar_ghi_8760h_20260923_223338.txt
